In [1]:
import subprocess
import sys

def pip(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True
    )

pip("ultralytics==8.3.*")

print("Installation complete.")
from pathlib import Path
import zipfile
import random
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import shutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.6 MB/s eta 0:00:00
Installation complete.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:

print("Searching for extracted best.pt contents...")

for p in Path("/kaggle/input").rglob("data.pkl"):
    print(p)

Searching for extracted best.pt contents...
/kaggle/input/datasets/tanvirmahtabtaneem/best-zip/best/data.pkl


In [3]:


data_files = list(
    Path("/kaggle/input").rglob("data.pkl")
)

best_dirs = []

for p in data_files:
    if p.parent.name == "best":
        best_dirs.append(p.parent)

if len(best_dirs) == 0:
    raise FileNotFoundError(
        "Could not find the extracted best model."
    )

if len(best_dirs) > 1:
    print("Multiple model folders found:")
    for p in best_dirs:
        print(p)

best_dir = best_dirs[0]

print("Using model folder:")
print(best_dir)

MODEL_PATH = Path(
    "/kaggle/working/best.pt"
)

with zipfile.ZipFile(
    MODEL_PATH,
    "w",
    compression=zipfile.ZIP_STORED
) as z:

    for file_path in best_dir.rglob("*"):

        if file_path.is_file():

            relative = file_path.relative_to(
                best_dir
            )

            archive_name = (
                Path("best") / relative
            )

            z.write(
                file_path,
                archive_name.as_posix()
            )

print("\nCreated:")
print(MODEL_PATH)

print(
    "Size:",
    MODEL_PATH.stat().st_size,
    "bytes"
)

Using model folder:
/kaggle/input/datasets/tanvirmahtabtaneem/best-zip/best

Created:
/kaggle/working/best.pt
Size: 40696293 bytes


In [4]:

model = YOLO(
    "/kaggle/working/best.pt"
)

print("===================================")
print("NEW FINE-TUNED MODEL LOADED")
print("===================================")

print(model.names)

NEW FINE-TUNED MODEL LOADED
{0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: 'Metro', 11: 'A', 12: 'Bha', 13: 'Cha', 14: 'Chha', 15: 'Da', 16: 'DA', 17: 'E', 18: 'Ga', 19: 'Gha', 20: 'Ha', 21: 'Ja', 22: 'Jha', 23: 'Ka', 24: 'Kha', 25: 'La', 26: 'Ma', 27: 'Na', 28: 'Pa', 29: 'Sa', 30: 'Sha', 31: 'Ta', 32: 'THA', 33: 'Tha', 34: 'U', 35: 'Bagerhat', 36: 'Bagura', 37: 'Bandarban', 38: 'Barguna', 39: 'Barisal', 40: 'Bhola', 41: 'Brahmanbaria', 42: 'Chandpur', 43: 'Chapainawabganj', 44: 'Chatto', 45: 'Chattogram', 46: 'Chuadanga', 47: "Cox's Bazar", 48: 'Cumilla', 49: 'Dhaka', 50: 'Dinajpur', 51: 'Faridpur', 52: 'Feni', 53: 'Gaibandha', 54: 'Gazipur', 55: 'Gopalganj', 56: 'Habiganj', 57: 'Jamalpur', 58: 'Jessore', 59: 'Jhalokati', 60: 'Jhenaidah', 61: 'Joypurhat', 62: 'Khagrachari', 63: 'Khulna', 64: 'Kishoreganj', 65: 'Kurigram', 66: 'Kustia', 67: 'Lakshmipur', 68: 'Lalmonirhat', 69: 'Madaripur', 70: 'Magura', 71: 'Manikganj', 72: 'Meherpur', 73: 'Moulvib

In [5]:
MODEL_PATH = Path(
    "/kaggle/working/best.pt"
)

In [6]:
# ============================================================
# 1. CHANGE ONLY THESE TWO PATHS
# ============================================================


# CHANGE THIS to the Kaggle input folder containing the
# company's raw plate images.
COMPANY_IMAGE_ROOT = Path(
    "/kaggle/input/datasets/tanvirmahtabtaneem/number-plate-test-images/images"
)

In [7]:
IMG_SIZE = 960

# Use 0.70 first.
# Lower to 0.60 only if too many tokens are missing.
CONF_THRESHOLD = 0.70

DEVICE = 0

EXPECTED_CLASSES = 102

OUTPUT_ROOT = Path(
    "/kaggle/working/company_pseudo_dataset"
)

IMAGE_OUTPUT = (
    OUTPUT_ROOT / "images"
)

LABEL_OUTPUT = (
    OUTPUT_ROOT / "labels"
)

ANNOTATED_OUTPUT = (
    OUTPUT_ROOT / "annotated"
)

SUMMARY_CSV = (
    OUTPUT_ROOT / "pseudo_label_summary.csv"
)

In [8]:
# ============================================================

def find_new_model():

    if MODEL_PATH.exists():
        return MODEL_PATH

    print(
        "Configured NEW_MODEL_PATH was not found."
    )

    print(
        "\nSearching /kaggle/input for "
        "bangla_lpdb_yolo11m_motion_finetune/weights/best.pt ..."
    )

    matches = list(
        Path("/kaggle/input").rglob(
            "bangla_lpdb_yolo11m_motion_finetune/weights/best.pt"
        )
    )

    if len(matches) == 1:
        print(
            "Found new model:",
            matches[0]
        )
        return matches[0]

    if len(matches) > 1:
        print(
            "\nMultiple new models found:"
        )

        for p in matches:
            print(p)

        raise RuntimeError(
            "Set NEW_MODEL_PATH manually because "
            "multiple fine-tuned models were found."
        )

    raise FileNotFoundError(
        "\nNew fine-tuned best.pt was not found.\n"
        "Change NEW_MODEL_PATH to the path of your "
        "saved fine-tuned best.pt."
    )


MODEL_PATH = find_new_model()



In [9]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}\n\n"
        "Change MODEL_PATH at the top of this file."
    )

if not COMPANY_IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Company image folder not found:\n"
        f"{COMPANY_IMAGE_ROOT}\n\n"
        "Change COMPANY_IMAGE_ROOT at the top of this file."
    )

In [10]:
#=============
# 4. CREATE OUTPUT FOLDERS
# ============================================================

IMAGE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

LABEL_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

ANNOTATED_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

In [11]:
# ============================================================
# 5. FIND COMPANY IMAGES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

image_paths = sorted(
    [
        p
        for p in COMPANY_IMAGE_ROOT.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
)

print("=" * 70)
print("COMPANY IMAGE CHECK")
print("=" * 70)
print("Company image root:", COMPANY_IMAGE_ROOT)
print("Images found:", len(image_paths))

if len(image_paths) == 0:
    raise RuntimeError(
        "No company images were found."
    )



COMPANY IMAGE CHECK
Company image root: /kaggle/input/datasets/tanvirmahtabtaneem/number-plate-test-images/images
Images found: 2145


In [12]:
# ============================================================
# 6. LOAD MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING CURRENT MODEL")
print("=" * 70)

model = YOLO(
    "/kaggle/working/best.pt"
)

print("===================================")
print("NEW FINE-TUNED MODEL LOADED")
print("===================================")

print(model.names)




LOADING CURRENT MODEL
NEW FINE-TUNED MODEL LOADED
{0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: 'Metro', 11: 'A', 12: 'Bha', 13: 'Cha', 14: 'Chha', 15: 'Da', 16: 'DA', 17: 'E', 18: 'Ga', 19: 'Gha', 20: 'Ha', 21: 'Ja', 22: 'Jha', 23: 'Ka', 24: 'Kha', 25: 'La', 26: 'Ma', 27: 'Na', 28: 'Pa', 29: 'Sa', 30: 'Sha', 31: 'Ta', 32: 'THA', 33: 'Tha', 34: 'U', 35: 'Bagerhat', 36: 'Bagura', 37: 'Bandarban', 38: 'Barguna', 39: 'Barisal', 40: 'Bhola', 41: 'Brahmanbaria', 42: 'Chandpur', 43: 'Chapainawabganj', 44: 'Chatto', 45: 'Chattogram', 46: 'Chuadanga', 47: "Cox's Bazar", 48: 'Cumilla', 49: 'Dhaka', 50: 'Dinajpur', 51: 'Faridpur', 52: 'Feni', 53: 'Gaibandha', 54: 'Gazipur', 55: 'Gopalganj', 56: 'Habiganj', 57: 'Jamalpur', 58: 'Jessore', 59: 'Jhalokati', 60: 'Jhenaidah', 61: 'Joypurhat', 62: 'Khagrachari', 63: 'Khulna', 64: 'Kishoreganj', 65: 'Kurigram', 66: 'Kustia', 67: 'Lakshmipur', 68: 'Lalmonirhat', 69: 'Madaripur', 70: 'Magura', 71: 'Manikganj', 72: '

In [13]:
# ============================================================
# 7. PROCESS IMAGES
# ============================================================

rows = []

print("\n" + "=" * 70)
print("GENERATING PSEUDO-LABELS")
print("=" * 70)

for index, image_path in enumerate(image_paths):

    results = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False
    )

    result = results[0]

    # Keep original image extension.
    output_image = (
        IMAGE_OUTPUT
        / image_path.name
    )

    # YOLO label uses the image stem.
    output_label = (
        LABEL_OUTPUT
        / f"{image_path.stem}.txt"
    )

    # Copy original image.
    shutil.copy2(
        image_path,
        output_image
    )

    detection_count = 0
    average_confidence = 0.0

    # Write YOLO labels.
    with open(
        output_label,
        "w",
        encoding="utf-8"
    ) as label_file:

        if result.boxes is not None:

            detection_count = len(
                result.boxes
            )

            if detection_count > 0:
                average_confidence = float(
                    result.boxes.conf.mean().item()
                )

            for box_index in range(
                detection_count
            ):

                class_id = int(
                    result.boxes.cls[
                        box_index
                    ].item()
                )

                # Safety check:
                # the public LPDB-A model has classes 0-101.
                if not (
                    0 <= class_id < EXPECTED_CLASSES
                ):
                    continue

                x_center, y_center, width, height = (
                    result.boxes.xywhn[
                        box_index
                    ]
                    .cpu()
                    .numpy()
                )

                confidence = float(
                    result.boxes.conf[
                        box_index
                    ].item()
                )

                label_file.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{width:.6f} "
                    f"{height:.6f} "
                    f"\n"
                )

    # Save visualized prediction.
    annotated = result.plot()

    annotated_path = (
        ANNOTATED_OUTPUT
        / image_path.name
    )

    cv2.imwrite(
        str(annotated_path),
        annotated
    )

    rows.append(
        {
            "image": image_path.name,
            "detected_tokens": detection_count,
            "average_confidence": average_confidence,
            "label_file": str(output_label)
        }
    )

    if (
        (index + 1) % 100 == 0
        or index == len(image_paths) - 1
    ):
        print(
            f"{index + 1}/{len(image_paths)} processed"
        )




GENERATING PSEUDO-LABELS
100/2145 processed
200/2145 processed
300/2145 processed
400/2145 processed
500/2145 processed
600/2145 processed
700/2145 processed
800/2145 processed
900/2145 processed
1000/2145 processed
1100/2145 processed
1200/2145 processed
1300/2145 processed
1400/2145 processed
1500/2145 processed
1600/2145 processed
1700/2145 processed
1800/2145 processed
1900/2145 processed
2000/2145 processed
2100/2145 processed
2145/2145 processed


In [14]:
# ============================================================
# 8. SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(rows)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print("Output folder:")
print(OUTPUT_ROOT)

print("\nImages:")
print(IMAGE_OUTPUT)

print("\nPseudo-labels:")
print(LABEL_OUTPUT)

print("\nAnnotated previews:")
print(ANNOTATED_OUTPUT)

print("\nSummary CSV:")
print(SUMMARY_CSV)

print("\nImages processed:", len(summary_df))
print(
    "Images with at least one detection:",
    int(
        (summary_df["detected_tokens"] > 0).sum()
    )
)

print(
    "Average tokens per image:",
    round(
        summary_df["detected_tokens"].mean(),
        2
    )
)

print(
    "Average token confidence:",
    round(
        summary_df["average_confidence"].mean(),
        4
    )
)

print(
    "\nNEXT STEP:"
)

print(
    "Manually correct the generated labels before "
    "using them for fine-tuning."
)



DONE
Output folder:
/kaggle/working/company_pseudo_dataset

Images:
/kaggle/working/company_pseudo_dataset/images

Pseudo-labels:
/kaggle/working/company_pseudo_dataset/labels

Annotated previews:
/kaggle/working/company_pseudo_dataset/annotated

Summary CSV:
/kaggle/working/company_pseudo_dataset/pseudo_label_summary.csv

Images processed: 2145
Images with at least one detection: 2139
Average tokens per image: 8.17
Average token confidence: 0.8133

NEXT STEP:
Manually correct the generated labels before using them for fine-tuning.
